# Protein ProGen Pretraining From Scratch

This notebook is protein-only. It reads compiled `train.txt` lines in the form `<|protein|>...<|endoftext|>`, builds the repo protein tokenizer from scratch, loads a ProGen backbone config through `libs`, adapts that config to the MDC decoder, then trains from random initialization.

In [ ]:
import json
import math
import sys
from pathlib import Path

import torch

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    raise RuntimeError("Could not locate project root from the current notebook directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.core import (
    MDCDecoderModel,
    build_mdc_config_from_progen_config,
    build_or_load_protein_tokenizer,
    build_or_load_protein_tokenizer_from_text_paths,
    build_progen_config,
    compute_mdc_causal_lm_loss,
    count_trainable_parameters,
    create_protein_lm_dataloader,
    create_streaming_protein_lm_dataloader,
    discover_protein_train_text_paths,
    evaluate_mdc_causal_lm_batch_loss,
    generate_protein_text,
    load_protein_corpus_text_parts,
    save_protein_pretrain_checkpoint,
    split_protein_corpus_text,
)

PROJECT_ROOT

In [ ]:
TRAIN_TEXT_PATH = PROJECT_ROOT / "data" / "compiled" / "refseq_bacteria_protein" / "train.txt"
TOKENIZER_MAP_PATH = TRAIN_TEXT_PATH.with_name("tokenizer_map.json")
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints" / "protein_from_scratch"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Local corpus can be one train.txt or many train_part_1.txt, train_part_2.txt, ... files.
TRAIN_PART_GLOB = "train_part_*.txt"
PREFER_LOCAL_TRAIN_PARTS = True
STREAM_LOCAL_TRAIN_PARTS = True

# Set this to an s3://bucket/prefix when the train parts live on MinIO/S3.
# The prefix can contain train_part_1.txt, train_part_2.txt, ... or any *.txt shards.
MINIO_TRAIN_PARTS_PREFIX_URI = ""
MINIO_TRAIN_PART_URIS = ()
TRAIN_PART_CACHE_DIR = PROJECT_ROOT / "data" / "cache" / "protein_train_parts"
KEEP_DOWNLOADED_TRAIN_PARTS = False

PROGEN_MODEL_SIZE = "0.8B"
PROGEN_CONFIG_OVERRIDES = {
    "emb_dim": 256,
    "n_heads": 4,
    "n_layers": 4,
    "hidden_dim": 1024,
    "head_dim": 64,
    "n_kv_groups": 2,
    "linear_key_head_dim": 64,
    "linear_value_head_dim": 64,
    "linear_num_key_heads": 4,
    "linear_num_value_heads": 4,
}
CONTEXT_LENGTH = 512
STRIDE = 256
TOKENIZER_VOCAB_SIZE = 512
REBUILD_TOKENIZER = False

TRAIN_RATIO = 0.9
BATCH_SIZE = 2
NUM_EPOCHS = 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP_NORM = 1.0
EVAL_FREQ = 50
EVAL_BATCHES = 10

USE_MINIO_TRAIN_PARTS = bool(MINIO_TRAIN_PARTS_PREFIX_URI or MINIO_TRAIN_PART_URIS)
LOCAL_TRAIN_TEXT_PATHS = ()
if TRAIN_TEXT_PATH.exists() or any(TRAIN_TEXT_PATH.parent.glob(TRAIN_PART_GLOB)):
    LOCAL_TRAIN_TEXT_PATHS = discover_protein_train_text_paths(
        TRAIN_TEXT_PATH,
        part_glob=TRAIN_PART_GLOB,
        prefer_parts=PREFER_LOCAL_TRAIN_PARTS,
    )
elif not USE_MINIO_TRAIN_PARTS:
    raise FileNotFoundError(
        f"Missing protein corpus: {TRAIN_TEXT_PATH} or {TRAIN_PART_GLOB} parts. "
        "Build it first with cmd/build_refseq_profile_text.py."
    )
elif not TOKENIZER_MAP_PATH.exists():
    raise FileNotFoundError(
        "When streaming training parts from MinIO without a local train.txt/train_part_*.txt corpus, "
        f"provide the tokenizer map first: {TOKENIZER_MAP_PATH}"
    )

config_summary = {
    "train_text_path": str(TRAIN_TEXT_PATH),
    "local_train_text_paths": [str(path) for path in LOCAL_TRAIN_TEXT_PATHS],
    "tokenizer_map_path": str(TOKENIZER_MAP_PATH),
    "minio_train_parts_prefix_uri": MINIO_TRAIN_PARTS_PREFIX_URI,
    "minio_train_part_uris": list(MINIO_TRAIN_PART_URIS),
    "stream_local_train_parts": STREAM_LOCAL_TRAIN_PARTS,
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "progen_model_size": PROGEN_MODEL_SIZE,
    "progen_config_overrides": PROGEN_CONFIG_OVERRIDES,
    "context_length": CONTEXT_LENGTH,
    "tokenizer_vocab_size": TOKENIZER_VOCAB_SIZE,
}
config_summary

In [ ]:
corpus_text = load_protein_corpus_text_parts(LOCAL_TRAIN_TEXT_PATHS) if LOCAL_TRAIN_TEXT_PATHS else ""
if LOCAL_TRAIN_TEXT_PATHS:
    tokenizer_artifact = build_or_load_protein_tokenizer_from_text_paths(
        LOCAL_TRAIN_TEXT_PATHS,
        tokenizer_map_path=TOKENIZER_MAP_PATH,
        vocab_size=TOKENIZER_VOCAB_SIZE,
        rebuild=REBUILD_TOKENIZER,
    )
else:
    tokenizer_artifact = build_or_load_protein_tokenizer(
        TRAIN_TEXT_PATH,
        tokenizer_map_path=TOKENIZER_MAP_PATH,
        vocab_size=TOKENIZER_VOCAB_SIZE,
        rebuild=REBUILD_TOKENIZER,
    )

if corpus_text:
    train_text, val_text = split_protein_corpus_text(corpus_text, train_ratio=TRAIN_RATIO)
else:
    train_text, val_text = "", ""

if USE_MINIO_TRAIN_PARTS:
    train_loader = create_streaming_protein_lm_dataloader(
        tokenizer_artifact.tokenizer,
        prefix_uri=MINIO_TRAIN_PARTS_PREFIX_URI or None,
        part_uris=MINIO_TRAIN_PART_URIS or None,
        cache_dir=TRAIN_PART_CACHE_DIR,
        keep_downloaded_parts=KEEP_DOWNLOADED_TRAIN_PARTS,
        context_length=CONTEXT_LENGTH,
        stride=STRIDE,
        batch_size=BATCH_SIZE,
        shuffle_parts=True,
        shuffle_examples=True,
    )
    train_loader_description = "minio streaming"
elif STREAM_LOCAL_TRAIN_PARTS and len(LOCAL_TRAIN_TEXT_PATHS) > 1:
    train_loader = create_streaming_protein_lm_dataloader(
        tokenizer_artifact.tokenizer,
        part_paths=LOCAL_TRAIN_TEXT_PATHS,
        context_length=CONTEXT_LENGTH,
        stride=STRIDE,
        batch_size=BATCH_SIZE,
        shuffle_parts=True,
        shuffle_examples=True,
    )
    train_loader_description = "local part streaming"
else:
    train_loader = create_protein_lm_dataloader(
        train_text,
        tokenizer_artifact.tokenizer,
        context_length=CONTEXT_LENGTH,
        stride=STRIDE,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    train_loader_description = "in-memory split"

val_loader = (
    create_protein_lm_dataloader(
        val_text,
        tokenizer_artifact.tokenizer,
        context_length=CONTEXT_LENGTH,
        stride=STRIDE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    if val_text
    else None
)

def dataloader_size(loader):
    if loader is None:
        return "disabled"
    try:
        return len(loader)
    except TypeError:
        return "streaming"

print(f"Corpus chars: {len(corpus_text):,}")
print(f"Local corpus files: {len(LOCAL_TRAIN_TEXT_PATHS)}")
print(f"Train loader: {train_loader_description}")
print(f"Tokenizer vocab: {tokenizer_artifact.vocab_size} rebuilt={tokenizer_artifact.rebuilt}")
print(f"Batches: train={dataloader_size(train_loader)} val={dataloader_size(val_loader)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
runtime_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

progen_config = build_progen_config(
    PROGEN_MODEL_SIZE,
    vocab_size=tokenizer_artifact.vocab_size,
    context_length=CONTEXT_LENGTH,
    dtype=runtime_dtype,
    overrides=PROGEN_CONFIG_OVERRIDES,
)

model_config = build_mdc_config_from_progen_config(
    progen_config,
    dtype=runtime_dtype,
    attention_pattern="as_config",
)
model = MDCDecoderModel(model_config).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(f"Device: {device} dtype={runtime_dtype}")
print(f"Trainable parameters: {count_trainable_parameters(model):,}")
model_config.to_dict()

In [ ]:
checkpoint_last_path = CHECKPOINT_DIR / "checkpoint_last.pt"
checkpoint_best_path = CHECKPOINT_DIR / "checkpoint_best.pt"
metrics_path = CHECKPOINT_DIR / "training_metrics.jsonl"

global_step = 0
tokens_seen = 0
train_losses = []
val_losses = []
best_val_loss = math.inf

def move_batch(batch, device):
    return type(batch)(
        input_ids=batch.input_ids.to(device),
        attention_mask=batch.attention_mask.to(device),
        labels=batch.labels.to(device),
    )

def append_metrics(payload):
    with metrics_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")

def save_checkpoint(path, epoch, val_loss):
    return save_protein_pretrain_checkpoint(
        path,
        model=model,
        optimizer=optimizer,
        model_config=model_config,
        tokenizer=tokenizer_artifact.tokenizer,
        tokenizer_map_path=tokenizer_artifact.tokenizer_map_path,
        epoch=epoch,
        global_step=global_step,
        tokens_seen=tokens_seen,
        train_losses=train_losses,
        val_losses=val_losses,
        best_val_loss=None if math.isinf(best_val_loss) else best_val_loss,
        training_args={
            "batch_size": BATCH_SIZE,
            "context_length": CONTEXT_LENGTH,
            "stride": STRIDE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "progen_model_size": PROGEN_MODEL_SIZE,
            "progen_config_overrides": PROGEN_CONFIG_OVERRIDES,
        },
        extra={
            "corpus_file": str(TRAIN_TEXT_PATH.resolve()),
            "corpus_files": [str(path.resolve()) for path in LOCAL_TRAIN_TEXT_PATHS],
            "minio_train_parts_prefix_uri": MINIO_TRAIN_PARTS_PREFIX_URI,
            "minio_train_part_uris": list(MINIO_TRAIN_PART_URIS),
            "progen_config": dict(progen_config),
            "last_eval_val_loss": val_loss,
        },
    )

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    for batch in train_loader:
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch.input_ids, attn_mask=batch.attention_mask)
        loss = compute_mdc_causal_lm_loss(logits, batch.labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        global_step += 1
        tokens_seen += int(batch.attention_mask.sum().item())

        if EVAL_FREQ > 0 and global_step % EVAL_FREQ == 0:
            train_eval_loss = evaluate_mdc_causal_lm_batch_loss(
                model, train_loader, device=device, max_batches=EVAL_BATCHES
            )
            val_loss = (
                evaluate_mdc_causal_lm_batch_loss(model, val_loader, device=device, max_batches=EVAL_BATCHES)
                if val_loader is not None
                else float("nan")
            )
            train_losses.append(train_eval_loss)
            val_losses.append(val_loss)
            append_metrics({
                "epoch": epoch,
                "global_step": global_step,
                "tokens_seen": tokens_seen,
                "train_loss": train_eval_loss,
                "val_loss": val_loss,
            })
            if val_loader is not None and val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(checkpoint_best_path, epoch, val_loss)
            print(f"epoch={epoch} step={global_step} train={train_eval_loss:.4f} val={val_loss:.4f}")

    val_loss = (
        evaluate_mdc_causal_lm_batch_loss(model, val_loader, device=device, max_batches=EVAL_BATCHES)
        if val_loader is not None
        else float("nan")
    )
    save_checkpoint(checkpoint_last_path, epoch, val_loss)

print(f"Saved last checkpoint: {checkpoint_last_path}")
print(f"Saved best checkpoint: {checkpoint_best_path if checkpoint_best_path.exists() else 'not yet'}")

In [ ]:
sample = generate_protein_text(
    model,
    tokenizer_artifact.tokenizer,
    prompt="<|protein|>",
    device=device,
    max_new_tokens=80,
    context_length=CONTEXT_LENGTH,
)
sample